# 02. Neo4j 지식 그래프 구축 — "Attention is All You Need"

`01_pdf_extraction.ipynb`에서 생성한 CSV 파일을 읽어 **Neo4j** 지식 그래프를 구축합니다.

## 그래프 스키마

```
[ Paper ] ←─PART_OF─ [ Section ] ─IS_NEXT_TO─▶ [ Section ]
                          │
                  IS_BELONGING_TO
                          ▼
                      [ Section ]
                          │
          ┌───────────────┼─────────────────┐
       REFERENCES     INTRODUCES        REFERENCES
          ▼               ▼                 ▼
       [ Table ]     [ Equation ]       [ Figure ]
          │                                 │
        SUPPORTS                       ILLUSTRATES
          ▼                                 ▼
       [ Section ]                     [ Section ]
```

## 노드 레이블
| Label | 속성 |
|-------|------|
| `Paper` | paper_id, title, authors, year, arxiv_id |
| `Section` | id, number, title, level, content, page, embedding |
| `Table` | id, number, caption, content, page, embedding |
| `Figure` | id, number, caption, description, page, embedding |
| `Equation` | id, number, formula, description, page, embedding |

## 관계 유형
| 관계 | 방향 |
|------|------|
| `IS_NEXT_TO` | Section→Section (같은 레벨, 순서) |
| `IS_BELONGING_TO` | Section→Section (자식→부모) |
| `CONTAINS` | Section→Section (부모→자식) |
| `PART_OF` | 모든 노드→Paper |
| `REFERENCES` | Section→Table/Figure/Equation |
| `INTRODUCES` | Section→Equation |
| `ILLUSTRATES` | Figure→Section |
| `SUPPORTS` | Table→Section |
| `COMPARES` | Table→Section |

## Step 0. 패키지 설치

In [1]:
!pip install neo4j openai python-dotenv pandas -q


[notice] A new release of pip is available: 24.0 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


## Step 1. 임포트 및 환경 설정

In [ ]:
import os
import time
import pandas as pd
from pathlib import Path
from dotenv import load_dotenv
from neo4j import GraphDatabase
from openai import OpenAI

# .env 로드
load_dotenv(".env", override=True)

# Neo4j 기본 접속 정보
NEO4J_URI      = os.getenv("NEO4J_URI",      "neo4j://127.0.0.1:7687")
NEO4J_USERNAME = os.getenv("NEO4J_USERNAME", "neo4j")
NEO4J_PASSWORD = os.getenv("NEO4J_PASSWORD", "12345678")
NEO4J_DATABASE = os.getenv("NEO4J_DATABASE", "neo4j")

# OpenAI
OPENAI_API_KEY    = os.getenv("OPENAI_API_KEY")
EMBEDDING_MODEL   = os.getenv("EMBEDDING_MODEL", "text-embedding-ada-002")

# 데이터 디렉터리
DATA_DIR = Path("data")

print(f"Neo4j URI      : {NEO4J_URI}")
print(f"Neo4j User     : {NEO4J_USERNAME}")
print(f"Neo4j Database : {NEO4J_DATABASE}")
print(f"Embedding Model: {EMBEDDING_MODEL}")
print(f"Data Directory : {DATA_DIR.resolve()}")

## Step 2. Neo4j 연결 테스트

In [ ]:
driver = GraphDatabase.driver(
    NEO4J_URI,
    auth=(NEO4J_USERNAME, NEO4J_PASSWORD),
)

# 연결 확인
driver.verify_connectivity()
print("✅ Neo4j 연결 성공")

# 간단한 Cypher로 버전 확인
with driver.session(database=NEO4J_DATABASE) as session:
    result = session.run("RETURN 'Hello Neo4j!' AS message, 1+1 AS two")
    record = result.single()
    print(f"   메시지: {record['message']}")
    print(f"   1+1  = {record['two']}")

## Step 3. CSV 파일 로드

`01_pdf_extraction.ipynb`에서 생성한 CSV 파일을 읽어들입니다.

In [ ]:
# 노드 데이터
sections_df   = pd.read_csv(DATA_DIR / "nodes_sections.csv")
tables_df     = pd.read_csv(DATA_DIR / "nodes_tables.csv")
figures_df    = pd.read_csv(DATA_DIR / "nodes_figures.csv")
equations_df  = pd.read_csv(DATA_DIR / "nodes_equations.csv")

# 관계 데이터
hierarchy_df  = pd.read_csv(DATA_DIR / "relationships_hierarchy.csv")
sequence_df   = pd.read_csv(DATA_DIR / "relationships_sequence.csv")
references_df = pd.read_csv(DATA_DIR / "relationships_references.csv")

print("=== 로드된 데이터 요약 ===")
print(f"  Section  노드: {len(sections_df)}")
print(f"  Table    노드: {len(tables_df)}")
print(f"  Figure   노드: {len(figures_df)}")
print(f"  Equation 노드: {len(equations_df)}")
print(f"  IS_BELONGING_TO: {len(hierarchy_df)}")
print(f"  IS_NEXT_TO     : {len(sequence_df)}")
print(f"  참조 관계      : {len(references_df)}")

## Step 4. TransformerGraphBuilder 클래스 정의

`graph_build_tutorial.ipynb`의 `CreateGraph` 패턴을 따라 `TransformerGraphBuilder` 클래스를 정의합니다.

### 주요 Cypher 패턴
- **UNWIND**: Python 리스트를 Neo4j에 배치 전달
- **MERGE**: 중복 없이 노드/관계 생성
- **ON CREATE SET**: 처음 생성 시에만 속성 설정

In [ ]:
class TransformerGraphBuilder:
    """Transformer 논문 지식 그래프 구축 클래스."""

    PAPER_META = {
        "paper_id": "1706.03762",
        "title": "Attention Is All You Need",
        "authors": "Vaswani, Shazeer, Parmar, Uszkoreit, Jones, Gomez, Kaiser, Polosukhin",
        "year": 2017,
        "arxiv_id": "1706.03762v7",
        "venue": "NeurIPS 2017",
    }

    def __init__(self, driver, data_dir: Path, database: str = "neo4j"):
        self.driver   = driver
        self.data_dir = data_dir
        self.database = database

    def _run(self, cypher: str, params: dict = None):
        """단일 Cypher 실행 헬퍼."""
        with self.driver.session(database=self.database) as session:
            return session.run(cypher, params or {}).data()

    def _run_batched(self, cypher: str, rows: list, chunk_size: int = 500):
        """대용량 데이터를 청크로 나누어 UNWIND 패턴으로 실행합니다."""
        total = len(rows)
        for i in range(0, total, chunk_size):
            chunk = rows[i : i + chunk_size]
            with self.driver.session(database=self.database) as session:
                session.run(cypher, {"rows": chunk})
        print(f"    → {total}건 처리 완료")

    # ── 1. DB 초기화 ─────────────────────────────────────────────────
    def db_cleanup(self):
        """기존 노드와 관계를 모두 삭제합니다."""
        print("[1/10] 기존 데이터 삭제 중...")
        self._run("MATCH (n) DETACH DELETE n")
        print("    → 삭제 완료")

    # ── 2. 제약조건 및 인덱스 ─────────────────────────────────────────
    def create_constraints_indexes(self):
        """유니크 제약조건과 인덱스를 생성합니다."""
        print("[2/10] 제약조건 및 인덱스 생성 중...")
        stmts = [
            "CREATE CONSTRAINT paper_id   IF NOT EXISTS FOR (p:Paper)    REQUIRE p.paper_id  IS UNIQUE",
            "CREATE CONSTRAINT section_id IF NOT EXISTS FOR (s:Section)  REQUIRE s.id        IS UNIQUE",
            "CREATE CONSTRAINT table_id   IF NOT EXISTS FOR (t:Table)    REQUIRE t.id        IS UNIQUE",
            "CREATE CONSTRAINT figure_id  IF NOT EXISTS FOR (f:Figure)   REQUIRE f.id        IS UNIQUE",
            "CREATE CONSTRAINT eq_id      IF NOT EXISTS FOR (e:Equation) REQUIRE e.id        IS UNIQUE",
            "CREATE INDEX section_number  IF NOT EXISTS FOR (s:Section)  ON (s.number)",
            "CREATE INDEX section_level   IF NOT EXISTS FOR (s:Section)  ON (s.level)",
        ]
        for stmt in stmts:
            self._run(stmt)
        print(f"    → {len(stmts)}개 제약조건/인덱스 생성 완료")

    # ── 3. Paper 루트 노드 ────────────────────────────────────────────
    def load_paper(self):
        """Paper 루트 노드를 생성합니다."""
        print("[3/10] Paper 노드 생성 중...")
        self._run(
            """
            MERGE (p:Paper {paper_id: $paper_id})
            ON CREATE SET
                p.title    = $title,
                p.authors  = $authors,
                p.year     = $year,
                p.arxiv_id = $arxiv_id,
                p.venue    = $venue
            """,
            self.PAPER_META,
        )
        print("    → Paper 노드 생성 완료")

    # ── 4. Section 노드 ───────────────────────────────────────────────
    def load_sections(self, df: pd.DataFrame):
        """Section 노드를 MERGE로 생성합니다."""
        print("[4/10] Section 노드 생성 중...")
        rows = df.fillna("").to_dict("records")
        for row in rows:
            row["level"] = int(row["level"])
            row["page"]  = int(row["page"])
        cypher = """
        UNWIND $rows AS row
        MERGE (s:Section {id: row.id})
        ON CREATE SET
            s.number  = row.number,
            s.title   = row.title,
            s.level   = toInteger(row.level),
            s.content = row.content,
            s.page    = toInteger(row.page)
        """
        self._run_batched(cypher, rows)

    # ── 5. Table 노드 ─────────────────────────────────────────────────
    def load_tables(self, df: pd.DataFrame):
        """Table 노드를 MERGE로 생성합니다."""
        print("[5/10] Table 노드 생성 중...")
        rows = df.fillna("").to_dict("records")
        for row in rows:
            row["number"] = int(row["number"])
            row["page"]   = int(row["page"])
        cypher = """
        UNWIND $rows AS row
        MERGE (t:Table {id: row.id})
        ON CREATE SET
            t.number  = toInteger(row.number),
            t.caption = row.caption,
            t.content = row.content,
            t.page    = toInteger(row.page)
        """
        self._run_batched(cypher, rows)

    # ── 6. Figure 노드 ────────────────────────────────────────────────
    def load_figures(self, df: pd.DataFrame):
        """Figure 노드를 MERGE로 생성합니다."""
        print("[6/10] Figure 노드 생성 중...")
        rows = df.fillna("").to_dict("records")
        for row in rows:
            row["number"] = int(row["number"])
            row["page"]   = int(row["page"])
        cypher = """
        UNWIND $rows AS row
        MERGE (f:Figure {id: row.id})
        ON CREATE SET
            f.number      = toInteger(row.number),
            f.caption     = row.caption,
            f.description = row.description,
            f.page        = toInteger(row.page)
        """
        self._run_batched(cypher, rows)

    # ── 7. Equation 노드 ──────────────────────────────────────────────
    def load_equations(self, df: pd.DataFrame):
        """Equation 노드를 MERGE로 생성합니다."""
        print("[7/10] Equation 노드 생성 중...")
        rows = df.fillna("").to_dict("records")
        for row in rows:
            row["page"] = int(row["page"])
        cypher = """
        UNWIND $rows AS row
        MERGE (e:Equation {id: row.id})
        ON CREATE SET
            e.number      = row.number,
            e.formula     = row.formula,
            e.description = row.description,
            e.page        = toInteger(row.page)
        """
        self._run_batched(cypher, rows)

    # ── 8. 계층 관계 ──────────────────────────────────────────────────
    def load_hierarchy_rels(self, df: pd.DataFrame):
        """IS_BELONGING_TO 및 CONTAINS 관계를 생성합니다."""
        print("[8/10] IS_BELONGING_TO / CONTAINS 관계 생성 중...")
        rows = df.to_dict("records")
        cypher = """
        UNWIND $rows AS row
        MATCH (child:Section  {id: row.from_id})
        MATCH (parent:Section {id: row.to_id})
        MERGE (child)-[:IS_BELONGING_TO]->(parent)
        MERGE (parent)-[:CONTAINS]->(child)
        """
        self._run_batched(cypher, rows)

    # ── 9. 순서 관계 ──────────────────────────────────────────────────
    def load_sequence_rels(self, df: pd.DataFrame):
        """IS_NEXT_TO 관계를 생성합니다."""
        print("[9/10] IS_NEXT_TO 관계 생성 중...")
        rows = df.to_dict("records")
        cypher = """
        UNWIND $rows AS row
        MATCH (a:Section {id: row.from_id})
        MATCH (b:Section {id: row.to_id})
        MERGE (a)-[:IS_NEXT_TO]->(b)
        """
        self._run_batched(cypher, rows)

    # ── 10. 참조 관계 ─────────────────────────────────────────────────
    def load_reference_rels(self, df: pd.DataFrame):
        """REFERENCES / ILLUSTRATES / SUPPORTS / COMPARES / INTRODUCES 관계를 생성합니다."""
        print("[10/10] 참조 관계 생성 중...")
        label_map = {
            "Section": "Section",
            "Table":   "Table",
            "Figure":  "Figure",
            "Equation": "Equation",
        }
        # 관계 유형별로 분리하여 처리
        for rel_type, group in df.groupby("relationship"):
            rows = group.to_dict("records")
            # 동적 레이블은 Cypher에서 직접 지원되지 않으므로 from/to 레이블 조합별로 처리
            for from_label, to_label in group.groupby(["from_label", "to_label"]).groups.keys():
                sub = group[(group["from_label"] == from_label) & (group["to_label"] == to_label)]
                sub_rows = sub.fillna("").to_dict("records")
                cypher = f"""
                UNWIND $rows AS row
                MATCH (a:{from_label} {{id: row.from_id}})
                MATCH (b:{to_label}   {{id: row.to_id}})
                MERGE (a)-[r:{rel_type}]->(b)
                ON CREATE SET r.description = row.description
                """
                self._run_batched(cypher, sub_rows)

    # ── 11. PART_OF 관계 ──────────────────────────────────────────────
    def load_part_of_rels(self):
        """모든 노드를 Paper 루트 노드에 PART_OF 관계로 연결합니다."""
        print("[+] PART_OF 관계 생성 중...")
        labels = ["Section", "Table", "Figure", "Equation"]
        for label in labels:
            cypher = f"""
            MATCH (n:{label})
            MATCH (p:Paper {{paper_id: '1706.03762'}})
            MERGE (n)-[:PART_OF]->(p)
            """
            self._run(cypher)
        print("    → PART_OF 관계 생성 완료")

    # ── 12. 임베딩 생성 ───────────────────────────────────────────────
    def create_embeddings(self, openai_client: OpenAI, embedding_model: str):
        """각 노드의 텍스트로 OpenAI 임베딩을 생성하여 Neo4j에 저장합니다."""
        print("[+] 임베딩 생성 중... (OpenAI API 호출)")

        def embed_text(text: str) -> list[float]:
            """단일 텍스트에 대한 임베딩 벡터를 반환합니다."""
            text = text.strip().replace("\n", " ")[:8000]  # 토큰 제한
            response = openai_client.embeddings.create(
                input=[text],
                model=embedding_model,
            )
            return response.data[0].embedding

        def update_node_embedding(label: str, id_field: str, text_field: str):
            """특정 레이블의 모든 노드에 임베딩을 추가합니다."""
            result = self._run(f"MATCH (n:{label}) RETURN n.{id_field} AS id, n.{text_field} AS text")
            print(f"    {label}: {len(result)}개 노드 처리 중...")
            for record in result:
                if not record["text"]:
                    continue
                emb = embed_text(str(record["text"]))
                self._run(
                    f"MATCH (n:{label} {{{id_field}: $node_id}}) SET n.embedding = $embedding",
                    {"node_id": record["id"], "embedding": emb},
                )
                time.sleep(0.1)  # API rate limiting

        # 각 노드 유형별 임베딩 텍스트 설정
        update_node_embedding("Section",  "id", "content")      # 섹션 본문
        update_node_embedding("Table",    "id", "content")      # 표 내용
        update_node_embedding("Figure",   "id", "description")  # 그림 설명
        update_node_embedding("Equation", "id", "description")  # 수식 설명

        print("    → 임베딩 생성 완료")

    # ── 13. 벡터 인덱스 생성 ─────────────────────────────────────────
    def create_vector_index(self):
        """Neo4j 벡터 인덱스를 생성합니다 (의미 검색용)."""
        print("[+] 벡터 인덱스 생성 중...")
        index_configs = [
            ("section_embedding",  "Section",  "embedding"),
            ("table_embedding",    "Table",    "embedding"),
            ("figure_embedding",   "Figure",   "embedding"),
            ("equation_embedding", "Equation", "embedding"),
        ]
        for index_name, label, prop in index_configs:
            self._run(f"""
            CREATE VECTOR INDEX {index_name} IF NOT EXISTS
            FOR (n:{label}) ON (n.{prop})
            OPTIONS {{indexConfig: {{
                `vector.dimensions`: 1536,
                `vector.similarity_function`: 'cosine'
            }}}}
            """)
        print(f"    → {len(index_configs)}개 벡터 인덱스 생성 완료")

    # ── 전체 실행 ─────────────────────────────────────────────────────
    def run_all(
        self,
        sections_df,
        tables_df,
        figures_df,
        equations_df,
        hierarchy_df,
        sequence_df,
        references_df,
        openai_client: OpenAI,
        embedding_model: str,
    ):
        """전체 파이프라인을 순서대로 실행합니다."""
        print("=" * 55)
        print("  Transformer 논문 지식 그래프 구축 시작")
        print("=" * 55)
        t0 = time.time()

        self.db_cleanup()
        self.create_constraints_indexes()
        self.load_paper()
        self.load_sections(sections_df)
        self.load_tables(tables_df)
        self.load_figures(figures_df)
        self.load_equations(equations_df)
        self.load_hierarchy_rels(hierarchy_df)
        self.load_sequence_rels(sequence_df)
        self.load_reference_rels(references_df)
        self.load_part_of_rels()
        self.create_embeddings(openai_client, embedding_model)
        self.create_vector_index()

        elapsed = time.time() - t0
        print("=" * 55)
        print(f"  ✅ 완료! 소요 시간: {elapsed:.1f}초")
        print("=" * 55)

print("TransformerGraphBuilder 클래스 정의 완료")

## Step 5. 그래프 구축 실행

> ⚠️ **주의**: 이 셀을 실행하면 기존 Neo4j 데이터가 모두 삭제됩니다.

In [ ]:
# OpenAI 클라이언트 초기화
openai_client = OpenAI(api_key=OPENAI_API_KEY)

# 빌더 초기화 및 전체 파이프라인 실행
builder = TransformerGraphBuilder(
    driver=driver,
    data_dir=DATA_DIR,
    database=NEO4J_DATABASE,
)

builder.run_all(
    sections_df=sections_df,
    tables_df=tables_df,
    figures_df=figures_df,
    equations_df=equations_df,
    hierarchy_df=hierarchy_df,
    sequence_df=sequence_df,
    references_df=references_df,
    openai_client=openai_client,
    embedding_model=EMBEDDING_MODEL,
)

## Step 6. 그래프 검증 쿼리

구축된 그래프의 노드/관계 수와 구조를 확인합니다.

In [ ]:
# ─── 노드 수 확인 ─────────────────────────────────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    node_counts = session.run("""
        MATCH (n)
        RETURN labels(n)[0] AS label, count(n) AS count
        ORDER BY count DESC
    """).data()

print("=== 노드 수 ===")
total = 0
for row in node_counts:
    print(f"  {row['label']:15s}: {row['count']:3d} 개")
    total += row['count']
print(f"  {'합계':15s}: {total:3d} 개")

In [ ]:
# ─── 관계 수 확인 ─────────────────────────────────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    rel_counts = session.run("""
        MATCH ()-[r]->()
        RETURN type(r) AS relationship, count(r) AS count
        ORDER BY count DESC
    """).data()

print("=== 관계 수 ===")
total = 0
for row in rel_counts:
    print(f"  {row['relationship']:20s}: {row['count']:3d} 개")
    total += row['count']
print(f"  {'합계':20s}: {total:3d} 개")

In [ ]:
# ─── 섹션 계층 구조 확인 ──────────────────────────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    hierarchy = session.run("""
        MATCH (child:Section)-[:IS_BELONGING_TO]->(parent:Section)
        RETURN parent.number AS parent, parent.title AS parent_title,
               child.number  AS child,  child.title  AS child_title
        ORDER BY parent.number, child.number
    """).data()

print("=== IS_BELONGING_TO 관계 ===")
for row in hierarchy:
    print(f"  {row['child']:8s} ({row['child_title'][:30]}) → {row['parent']} ({row['parent_title'][:25]})")

In [ ]:
# ─── IS_NEXT_TO 관계 확인 ─────────────────────────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    seq = session.run("""
        MATCH (a:Section)-[:IS_NEXT_TO]->(b:Section)
        RETURN a.number AS from_num, a.title AS from_title,
               b.number AS to_num,   b.title AS to_title
        ORDER BY a.number
    """).data()

print("=== IS_NEXT_TO 관계 (최상위 섹션) ===")
for row in seq:
    if '.' not in str(row['from_num']):  # 최상위만 표시
        print(f"  {row['from_num']} {row['from_title'][:25]} → {row['to_num']} {row['to_title'][:25]}")

print("\n=== IS_NEXT_TO 관계 (하위 섹션) ===")
for row in seq:
    if '.' in str(row['from_num']):
        print(f"  {row['from_num']:8s} → {row['to_num']:8s}")

In [ ]:
# ─── 임베딩 생성 확인 ─────────────────────────────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    emb_check = session.run("""
        MATCH (n)
        WHERE n.embedding IS NOT NULL
        RETURN labels(n)[0] AS label,
               count(n) AS nodes_with_embedding,
               size(n.embedding) AS embedding_dim
        ORDER BY label
    """).data()

print("=== 임베딩 현황 ===")
for row in emb_check:
    print(f"  {row['label']:12s}: {row['nodes_with_embedding']}개 노드, 차원수={row['embedding_dim']}")

In [ ]:
# ─── 특정 섹션과 연결된 그림/표/수식 확인 ─────────────────────────
with driver.session(database=NEO4J_DATABASE) as session:
    related = session.run("""
        MATCH (s:Section)-[r]->(n)
        WHERE s.number = '3'
        RETURN s.title AS section,
               type(r) AS relationship,
               labels(n)[0] AS target_type,
               n.id AS target_id
    """).data()

print("=== 섹션 3 (Model Architecture) 연결 노드 ===")
for row in related:
    print(f"  [{row['section']}] --{row['relationship']}--> [{row['target_type']}] {row['target_id']}")

In [ ]:
# ─── Neo4j Browser 시각화용 Cypher ────────────────────────────────
print("=" * 60)
print("Neo4j Browser에서 실행할 시각화 Cypher 쿼리")
print("=" * 60)
print("""
// 전체 그래프 개요 (50개 노드)
MATCH (n)-[r]->(m) RETURN n, r, m LIMIT 50

// 섹션 계층 구조만
MATCH (a:Section)-[:IS_BELONGING_TO]->(b:Section)
RETURN a, b

// 섹션 3과 연결된 모든 노드
MATCH (s:Section {number: '3'})-[r*1..2]-(n)
RETURN s, r, n

// Paper에서 시작하는 전체 경로
MATCH p=(paper:Paper)<-[:PART_OF]-(n)
RETURN p LIMIT 30
""")

print("\n✅ 다음 단계: 03_graphrag.ipynb 실행")